# 実行環境：使っているPythonを確認する

対応する本文は [Python実行環境の構築](../chapters/setup.tex) である。このNotebookを開けている状態から、実際に使われているPythonを調べる。環境を最初から導入する手順は本文の冒頭にある。

本文の問1はuv、問2はvenvによる比較である。ここでは最初に今の環境とvenvを確かめ、その後でuvの手順を読む。OSごとに必要な端末は、macOSではターミナル、WindowsではWSLのUbuntuである。

## 今動いているPythonで計算する

次のセルをShift+Enterで実行する。計算結果の5と、Pythonの版、実行ファイルの場所が表示される。実行ファイルは、コードを動かしているPython本体である。

In [1]:
import sys

print(3 + 2)
print("Python", sys.version.split()[0])
print("実行ファイル:", sys.executable)

5
Python 3.12.14
実行ファイル: [教材フォルダー]/.venv/bin/python


## 問2：別の作業用環境を作る

仮想環境は、その作業で使う追加ライブラリを別の作業と分けて保持するためのものである。まず小さな実習用環境を作り、そのPythonを直接指定して起動する。

端末で行う基本操作は `python -m venv .venv` である。`-m` はPythonに含まれるvenvの機能を実行する指定、末尾の `.venv` は作成先のフォルダー名である。ここではNotebook用の環境と区別するため、専用の出力先へ作る。

### 保存先を用意する

この後のセルではファイルを作る。次のセルを実行して、`notebooks/_outputs/02_setup/` を用意する。表示された場所に結果が保存され、同名の実習用ファイルは再実行すると置き換わる。

Notebookは教材の `notebooks` フォルダー、または教材の一番上のフォルダーから実行する。`Path.cwd()` は現在のフォルダーを調べ、`mkdir()` は保存先を作る。

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "main.tex").is_file():
    raise FileNotFoundError("教材のフォルダーか notebooks フォルダーから実行してください。")
WORK = ROOT / "notebooks" / "_outputs" / "02_setup"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("保存先:", WORK.relative_to(ROOT))

Python 3.12.14
保存先: notebooks/_outputs/02_setup


次のセルは実習用のPythonを起動し、その環境の場所を調べる。`prefix` は今使っている環境のフォルダー、`base_prefix` は元のPythonのフォルダーである。表示された `prefix` が作成先と一致し、`base_prefix` と異なることを確認する。`assert` は条件が違うとエラーで止まる確認用の命令である。表示が「作成したvenvから起動できました」となれば、比較条件を満たしている。

pipはこの小さな実習用環境には導入しない。これはPythonの切り替えだけを確かめるセルであり、ライブラリの導入は本文の手順で行う。

In [3]:
import json
import subprocess
import venv

ENV_DIR = WORK / "venv_example"
venv.EnvBuilder(with_pip=False).create(ENV_DIR)
python_in_venv = ENV_DIR / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
probe = "import sys,json; print(json.dumps({'executable':sys.executable,'prefix':sys.prefix,'base_prefix':sys.base_prefix}))"
result = subprocess.run([str(python_in_venv), "-c", probe], capture_output=True, text=True, check=True)
info = json.loads(result.stdout)
print("作成先:", ENV_DIR)
for key, value in info.items():
    print(key, ":", value)
assert Path(info["prefix"]).resolve() == ENV_DIR.resolve()
assert info["prefix"] != info["base_prefix"]
print("作成した venv から起動できました。")

作成先: [教材フォルダー]/notebooks/_outputs/02_setup/venv_example
executable : [教材フォルダー]/notebooks/_outputs/02_setup/venv_example/bin/python
prefix : [教材フォルダー]/notebooks/_outputs/02_setup/venv_example
base_prefix : [Python本体]
作成した venv から起動できました。


### 端末で有効化と解除を比べる

別の作業フォルダーで、次を順に実行する。`source` はその端末で使うPythonを切り替え、`deactivate` は切り替える前へ戻す。Notebookのセルを動かすPythonは、別の端末で `source` を実行しただけでは切り替わらない。

```sh
python -m venv .venv
command -v python
source .venv/bin/activate
command -v python
python --version
python -m pip --version
deactivate
command -v python
```

有効化後は `command -v python` とpipの表示に、同じ `.venv` の場所が含まれることを確認する。解除後は有効化前の場所へ戻る。

## 問1：uvでプロジェクトを用意する

複数のライブラリを使う作業では、必要なものの名前をファイルに記録すると、別の環境でもそろえやすい。uvはこの記録と導入をまとめて行う。

端末で次を実行する。`usgs_project` は新しく作るフォルダーの名前である。

```sh
uv init --package usgs_project
cd usgs_project
uv add numpy matplotlib scipy
uv run python --version
uv run python -c "import sys, numpy, matplotlib, scipy; print(sys.executable)"
```

`uv add` は必要なライブラリを記録し、プロジェクトの環境へ導入する。`uv run` はその環境で命令を実行する。最後の行に `.venv` 以下のPythonの場所が表示されれば、そのプロジェクトの環境を使っている。

`.python-version` は使うPythonの版、`pyproject.toml` は作業名や必要なライブラリ、`uv.lock` は解決した具体的なライブラリの版を記録する。`--package` は `src/` にコードを置く構成を指定する。uv 0.12より前と以降で既定構成が違っても、この指定でそろえられる。

次のセルでは、このNotebookのPythonから3つのライブラリを読み込めることを確かめる。これは端末で作るuvプロジェクトとは別の確認である。

In [4]:
from importlib import import_module

print("実行ファイル:", sys.executable)
print("仮想環境:", sys.prefix != sys.base_prefix)
for name in ["numpy", "matplotlib", "scipy"]:
    print(name, import_module(name).__version__)

実行ファイル: [教材フォルダー]/.venv/bin/python
仮想環境: True
numpy 2.5.4


matplotlib 3.11.2
scipy 1.18.1


## 問3：pyenv の shim と実体を区別する

`command -v python` が示す shim は中継用の実行ファイルである。`pyenv which python` は pyenv が選んだ実体を示す。2系統の Python を導入済みの端末で、別々のフォルダーに移動して次を比較する。使用する版は `pyenv versions` にあるものを選ぶ。

```sh
pyenv local 3.11.11
python --version
command -v python
pyenv which python
cat .python-version
python -c "import sys; print(sys.executable); print(sys.version)"
```

もう一方で別の版を設定する。`uv run python --version` はプロジェクト環境の選択なので、shim の表示だけで一致を判断しない。Notebook の実行だけでは親の対話シェルの有効化や pyenv の切り替えは検証できない。

## 補足：設定値を読み戻す

同じ解析を繰り返すときは、時間窓などを設定ファイルへ保存しておくと確認しやすい。JSONは、名前と値の組を文字列として保存する形式である。次のセルは `coincidence_window_ns`（時間窓 [ns]）を250として保存し、読み戻して同じ値になることを確かめる。JSONの詳しい読み書きは入出力の章で扱う。

`type(window_ns) is int` は、値の種類が整数かを調べる。Pythonでは `True` と `False` も整数との比較に使えるため、ここでは整数型そのものを確認してから、0以上という条件を調べる。

In [5]:
config_path = WORK / "config.json"
config_path.write_text(json.dumps({"coincidence_window_ns": 250, "figure_format": "pdf"}, indent=2), encoding="utf-8")
config = json.loads(config_path.read_text(encoding="utf-8"))
window_ns = config["coincidence_window_ns"]
assert type(window_ns) is int and window_ns >= 0
print(config)

{'coincidence_window_ns': 250, 'figure_format': 'pdf'}
